# **Lab 09: Lapse rates and stability**

In this lab, we will practice calculating lapse rates and identifying stability in the ocean (Part 1) and the atmosphere (Parts 2 and 3).

**Learning Target 14**: I can determine environmental stability in the ocean and atmosphere by comparing density/temperature profiles to adiabatic reference paths.

## How to submit this lab

1. Check your code runs cleanly. In the top menu, click **Kernel > Restart Kernel and Run All Cells**.
2. Save your notebook by pressing `Cmd + S` on Mac or `Ctrl + S` on Windows, or by going to **File > Save Notebook**.
3. Go to **File > Save and export notebook as > PDF**. This should start a download of your lab in PDF form.
4. Open your PDF file to make sure everything looks right, and upload it to the Canvas assignment for today.
5. Make sure to close your kernels on the JupyterHub. On the top bar, click **Kernel > Shut Down All Kernels...** and confirm.

# Setup

Check that you are running with the aos330 environment.

In [ ]:
# Import core packages using standard nicknames
import pandas as pd
import matplotlib.pyplot as plt

import os

os.environ["PROJ_LIB"] = "/opt/anaconda3/envs/aos330/share/proj"
os.environ["PROJ_DATA"] = "/opt/anaconda3/envs/aos330/share/proj"

from metpy.plots import SkewT
from metpy.units import units

# defining constants you will need later

g = 9.81  # acceleration due to gravity [m s^-2]

Rd = 287  # specific gas constant for dry air [J K^-1 kg^-1]
Rv = 461.5  # specific gas constant for water vapor [J K^-1 kg^-1]

cv = 718  # specific heat capacity at constant volume for dry air [J K^-1 kg^-1]
cp = 1005  # specific heat capacity at constant pressure for dry air [J K^-1 kg^-1]

eps = Rd / Rv  # ratio of dry air gas constant to water vapor gas constant
kappa = Rd / cp  # ratio of gas constant to heat capacity

# Part 1: Stability in the ocean (10 pts)

In the first part of this lab, you will look at ARGO floats to explore stability ($E=-\frac{1}{\rho}\frac{\partial \rho}{\partial z}$) in the upper ocean.

## 1.1 Read in data

We will start by reading and plotting a single profile from the Mediterranean on 2026-09-23.

The columns of the data are:
- `PRES`: in-situ pressure in decibars (*remember 1 dbar is approximately 1 m of depth in ocean*)
- `SA`: absolute salinity in ppt (g salt / kg water)
- `TEMP`: in-situ temperature in degrees Celsius
- `RHO`: in-situ density calculated using TEOS-10 (not our linearized seawater equation of state) in kg/m3
- `E`: stability parameter in units 1/m


In [ ]:
# read in ARGO data as a dataframe
site = "mediterranean_20260923"
df = pd.read_csv(f"data/ocean_profile_{site}.csv")

# print out the first 5 rows of the dataframe
print(df.head(5))

The following code block will plot the vertical profile in our sample region, with four panels corresponding to temperature, salinity, density, and stability.

In [ ]:
# set up axes
fig, axes = plt.subplots(1, 4, sharey=True, figsize=(15, 5))

# plot each panel
df.plot(ax=axes[0], x="TEMP", y="PRES", label=site, grid=True)
df.plot(ax=axes[1], x="SA", y="PRES", label=site, grid=True)
df.plot(ax=axes[2], x="RHO", y="PRES", label=site, grid=True)
df.plot(ax=axes[3], x="E", y="PRES", label=site, grid=True)

# label x-axes
axes[0].set_xlabel("Temperature (degC)")
axes[1].set_xlabel("Salinity (ppt)")
axes[2].set_xlabel("Density (kg/m3)")
axes[3].set_xlabel("Stability (1/m)")


# label Y-axis
axes[0].set_ylabel("Pressure (dbar)")

# Invert y-axis
axes[0].invert_yaxis()

### Question 1 (1 pt)

Based on the above plot, at which depth is the profile most stable? Describe what you looked at on the plot to determine this. 

[0.5 pt for reasonable depth estimate, 0.5 pt for description]

### Answer 1

#TODO: *(Type your answer here)*

## 1.2 Buoyancy calculations

To explore stability further, let's apply the parcel method to determine the static stability at 30 dbar.


We will need to be able to look up the density at a given pressure value. We'll set up `rho`, which is a type of object called a "Pandas Series". Notice this looks similar to our dataframe from earlier, except we're only looking at density.

In [ ]:
# No need to edit this

# set up a Pandas Series where the index is pressure and the values are density
rho = df.set_index("PRES")["RHO"]

print(rho)

To output the density value at a specific pressure level, use: `rho[p]` where `p` is the value you are looking for. 

Practice by outputting the density at 30dbar below. You should get a value around 1026.51 kg/m3.

In [ ]:
#TODO: Your code here


Visually, this is what we are doing: We take a parcel starting at 30dbar (blue circle), and move it up by a small amount (orange circle). For now, say we move it by 10dbar so that the parcel is now at 20dbar. 

In [ ]:
fig, ax = plt.subplots(1, 1, sharey=True, figsize=(5, 5))

# plot each panel
df.plot(ax=ax, x="RHO", y="PRES", label=site, grid=True)

# plot our parcel as a circle
ax.scatter(rho[30], 30)

# displace our parcel upward by 10 dbar
ax.scatter(rho[30], 30 - 10)

# displace our parcel downward by 10 dbar
ax.scatter(rho[30], 30 + 10)

# label x-axes
ax.set_xlabel("Density (kg/m3)")

# label Y-axis
ax.set_ylabel("Pressure (dbar)")

# Invert y-axis
ax.invert_yaxis()

# set limits to zoom in on our point
ax.set_ylim(60, 0)
ax.set_xlim(1025, 1028)

### Question 2 (1 pt)

Calculate the buoyancy of the parcel at its new location.

To do this:
1. Save the original density of the parcel as `rho_parcel`. We are assuming the parcel maintains its original density as we move it (i.e., adiabatic compression term is very small).
2. Retrieve the density of the environment at the parcel's new pressure (using `rho[p_new]`) and save this as `rho_env`.
3. Use our formula for buoyant force $f_B$ to calculate the sign and magnitude of the buoyant force and save this as `f_B`.

[1 pt for correct value and sign]

#### Answer 2

In [ ]:
#TODO: Your code here


# No need to edit this line, should work if you named your variables correctly.
print(f"The buoyant force is {f_B:.3f} m s^-2.")

### Question 3 (1 pt)

1. What direction is the buoyant force on the parcel? 
2. Describe what would happen to the parcel that we displaced upwards: where would it go?

[0.5 pts each]

#### Answer 3

#TODO: *(Type your answer here)*

### Question 4 (1 pt)

Without rerunning your code, describe what would happen if we now displaced the parcel **downward** by 10dbar (moving it from the blue dot to the green dot on your figure from earlier).

1. Describe the direction of the buoyant force 
2. What would happen to the parcel we displaced downwards?

#### Answer 4

#TODO: *(Type your answer here)*

### Question 5 (1 pt)

1. What do your answers in Questions 3 and 4 tell us about the static stability of the profile at 30dbar? 
2. Describe how 30dbar looks on the stability plot from Section 1.1. Does this agree with your answer? Focus on the sign and magnitude of the stability parameter $E$ at this depth.

#### Answer 5

#TODO: *(Type your answer here)*

## 1.3 Comparing stability

For the last part of Part 1 (ocean), we'll work on comparing the stability of different profiles. 

The following code block plots two consecutive profiles from the Labrador Sea near Greenland. As the Greenland ice sheet melts over the summer, it dumps freshwater into the surrounding ocean, which makes the water near the surface fresher and cooler (remember that it takes heat to melt ice). 

Remember from our Module on the equation of state that cooling (drop in temperature) and freshening (increase in salinity) have **opposite** impacts on the density of seawater.

Let's see what the net impact is on density and how this process affects the stability of the near-surface ocean.

In [ ]:
# set up axes
fig, axes = plt.subplots(1, 4, sharey=True, figsize=(15, 5))

for site in ["labradorsea_20260729", "labradorsea_20260808"]:
    # read in dataframe
    df = pd.read_csv(f"data/ocean_profile_{site}.csv")

    # plot each panel
    df.plot(ax=axes[0], x="TEMP", y="PRES", label=site, grid=True)
    df.plot(ax=axes[1], x="SA", y="PRES", label=site, grid=True)
    df.plot(ax=axes[2], x="RHO", y="PRES", label=site, grid=True)
    df.plot(ax=axes[3], x="E", y="PRES", label=site, grid=True)

# label x-axes
axes[0].set_xlabel("Temperature (degC)")
axes[1].set_xlabel("Salinity (ppt)")
axes[2].set_xlabel("Density (kg/m3)")
axes[3].set_xlabel("Stability (1/m)")


# label Y-axis
axes[0].set_ylabel("Pressure (dbar)")


# Invert y-axis
axes[0].invert_yaxis()

### Question 6 (1 pt)

1. Describe how the freshening and cooling of near-surface water impacts the density profile based on the plot. 

    *Hint: describe whether density increases or decreases, and what depths are affected*

2. Based on this, do you think that salinity or temperature had a bigger impact on the density profile?

[0.5 pts each]

#### Answer 6

#TODO: *(Type your answer here)*

### Question 7 (2 pts)

In both profiles, calculate and compare the buoyant force of a parcel at 25 dbar that is displaced to 35 dbar. 

Use the Series named `rho1` (pre-melting) and `rho2` (post-melting) below. You may find it useful to copy your code from Questions 2-4.

Name the buoyant forces `f_B1` and `f_B2` respectively.

[1 pt each for calculation per parcel]

In [ ]:
# Late July profile, before melting
rho1 = pd.read_csv(f"data/ocean_profile_labradorsea_20260729.csv").set_index("PRES")[
    "RHO"
]

# Early August profile, after melting
rho2 = pd.read_csv(f"data/ocean_profile_labradorsea_20260808.csv").set_index("PRES")[
    "RHO"
]

In [ ]:
#TODO: Your code here


# No need to edit this line, should work if you named your variables correctly.
print(
    f"The buoyant force of the parcel from the pre-melting profile is {f_B1:.4f} m s^-2."
)
print(
    f"The buoyant force of the parcel from the post-melting profile is {f_B2:.4f} m s^-2."
)

### Question 8 (2 pts)

1. Compare the magnitude of the restoring buoyant force between the two profiles. Which displaced profile will be returned to its original position more strongly?

2. Does your answer agree with the relative stability of the two profiles? Relate your answer to both the strength of the density gradient and the stability parameter $E$ plotted above?

[1 pt each]

#### Answer 8

#TODO: *(Type your answer here)*

# Part 2: Stability in the atmosphere (15 pts)

To determine stability in the upper ocean, where we assumed the density of our parcel did not change very much as we lifted/lowered it. 

On the other hand, in the atmosphere, we need to consider changes in the density of our parcel as we lift/lower it adiabatically. Instead of just looking at the sign of the density gradient in the environment then, we also need to compare its magnitude to the change in density due to adiabatic expansion/compression. 

There are a few different ways to determine stability in the atmosphere. We will explore two of them today.

We will start by reading in a sounding from Luanda, Angola (this is one of the soundings you plotted way back in Lab02) and plotting it on a Skew-T.

In [ ]:
# read in the dataframe with our sounding
df = pd.read_csv(f"data/atmos_sounding_luanda_20260401_12Z.csv")

# intialize figure
fig = plt.figure(figsize=(9, 9))

# initialize the Skew-T axis
skew = SkewT(fig, rotation=45)

# plot temperature and pressure on Skew-T
skew.ax.plot(df.temperature, df.pressure, color="black")

# draw dry adiabats for reference
skew.plot_dry_adiabats()

# axis labels
skew.ax.set_xlabel("Temperature (°C)")
skew.ax.set_ylabel("Pressure (hPa)")

# adding temperature labels at the top of our plot to help read values
skew.ax.tick_params(
    axis="x",
    top=True,
    labeltop=True,
    bottom=True,
    labelbottom=True,
)

## 2.1 Dry atmosphere (Parcel method)

Remember that in the dry atmosphere, for a given pressure level, density is directly proportional to temperature due to the equation of state. 

This makes our lives much easier since we can directly measure temperature and read it on a a Skew-T!

### Question 9 (1 pt)

Use the Skew-T to estimate the temperature of the profile at 700 hPa, and save that value as `T_init`.

Use `skew.ax.scatter(T_init, p_init)` to plot a circle at your estimated value. This circle will be our parcel's initial position.

*Hint: You can try moving around your `T_init` value to align it with the sounding profile. Getting it right to within ±1 degC is sufficient for this lab.*

#### Answer 9

In [ ]:
# intialize figure
fig = plt.figure(figsize=(9, 9))

# initialize the Skew-T axis
skew = SkewT(fig, rotation=45)

# plot temperature and pressure on Skew-T
skew.ax.plot(df.temperature, df.pressure, color="black")

# plot your estimated point here

#TODO: Your code here


# draw dry adiabats for reference
skew.plot_dry_adiabats()

# axis labels
skew.ax.set_xlabel("Temperature (°C)")
skew.ax.set_ylabel("Pressure (hPa)")

# adding temperature labels at the top of our plot to help read values
skew.ax.tick_params(
    axis="x",
    top=True,
    labeltop=True,
    bottom=True,
    labelbottom=True,
)

Most of this lab will involve lifting/lowering parcels dry adiabatically. To help, I've written you a function called `lift_dry_adiabatically` to calculate the new temperature of a parcel that is lifted/lowered dry adiabatically, following Poisson's equation.

The inputs are the initial temperature, and initial and final pressures. The function takes degC and hPa, so we can use it more easily with our Skew-T.


In [ ]:
def lift_dry_adiabatically(T, p, p_new):
    # Calculate the new temperature (T_new) [degC] as a function of initial pressure (p) [hPa], final pressure (p_new), and initial temperature (T) [degC] for dry adiabatic lifting

    T_new = ((T + 273.15) * (p_new / p) ** kappa) - 273.15

    return T_new

### Question 10 (1 pt)

Use  `lift_dry_adiabatically` to compute the new temperature of your parcel after lifting it dry adiabatically by 50 hPa (from 700 to 650 hPa) and save it as a variable named `T_parcel`. To verify that your function is working properly, compare your value to what you would get using the Skew-T and following the dry adiabat.

#### Answer 10

In [ ]:
p_parcel = p_init - 50

#TODO: Your code here


### Question 11 (1 pt)

1. What is the sounding temperature at 650 hPa? This is the **environmental** temperature. Save this as a variable named `T_env`.
2. How does this compare to your lifted parcel's temperature (from Question 10)? This is the **parcel** temperature `T_parcel`.

#### Answer 11

#TODO: *(Type your answer here)*

In [ ]:
# save your environmental temperature as T_env

#TODO: Your code here


Visually, this is what we are doing: We move the parcel (blue dot) upwards and bring it along the dry adiabat to the new position (orange dot).

In [ ]:
# intialize figure
fig = plt.figure(figsize=(9, 9))

# initialize the Skew-T axis
skew = SkewT(fig, rotation=45)

# plot temperature and pressure on Skew-T
skew.ax.plot(df.temperature, df.pressure, color="black")

# original parcel
skew.ax.scatter(T_init, p_init, zorder=10)


# lifted_parcel
skew.ax.scatter(T_parcel, p_parcel, zorder=10)

# draw dry adiabats for reference
skew.plot_dry_adiabats()

# axis labels
skew.ax.set_xlabel("Temperature (°C)")
skew.ax.set_ylabel("Pressure (hPa)")

# adding temperature labels at the top of our plot to help read values
skew.ax.tick_params(
    axis="x",
    top=True,
    labeltop=True,
    bottom=True,
    labelbottom=True,
)

### Question 12 (1 pt)

Assuming the parcel is totally dry, use `T_env` and `T_parcel` to calculate the buoyant force, including its sign.


#### Answer 12

In [ ]:
#TODO: Your code here


# No need to edit this line, should work if you named your variables correctly.
print(f"The buoyant force is {f_B:.3f} m s^-2.")

### Question 13 (1 pt)

1. What is the direction of the buoyant force?
2. What does this mean for the static stability at 700 hPa?

[0.5 each]

#### Answer 13

#TODO: *(Type your answer here)*

## 2.2 Dry atmosphere (Skew-T method)

Another convenient way to determine the stability of a profile is to use a Skew-T (yet another reason why Skew-T's are handy!).

To do this, compare the slope of the environmental profile (black line) to the slope of the dry adiabat (red dashed line). This is a graphical way of comparing the environmental lapse rate to the dry adiabatic lapse rate.

*Note: I will draw many dry adiabats on this plot to make your life easier for now, but when you use real skew-Ts, you often have to interpolate between the dry adiabats to estimate.*

In [ ]:
# intialize figure
fig = plt.figure(figsize=(9, 9))

# initialize the Skew-T axis
skew = SkewT(fig, rotation=45)

# plot temperature and pressure on Skew-T
skew.ax.plot(df.temperature, df.pressure, color="black")


# draw dry adiabats for reference
skew.plot_dry_adiabats(np.arange(-30, 90, 5) * units("degC"))

# axis labels
skew.ax.set_xlabel("Temperature (°C)")
skew.ax.set_ylabel("Pressure (hPa)")

# adding temperature labels at the top of our plot to help read values
skew.ax.tick_params(
    axis="x",
    top=True,
    labeltop=True,
    bottom=True,
    labelbottom=True,
)

### Question 14 (1 pt)

Does the environmental profile cool more or less with height than the dry adiabat? What does this tell us about the (dry) static stability of the profile at 700 hPa? 

#### Answer 14

#TODO: *(Type your answer here)*

### Question 15 (2 pts)

One of the benefits of the Skew-T method is that (with some practice) it makes it quick to read the stability at several points in the profile!

Practice by using the Skew-T to find: 
1. one pressure level where the profile is unstable to dry adiabatic motions. 
2. one pressure level where the profile is (close to) neutral for dry adiabatic motions.

#### Answer 15

#TODO: *(Type your answer here)*

## 2.4 Moist atmosphere

Previously, we've only been considering a dry atmosphere. However, remember that a saturated parcel cools at a different rate with height than a dry parcel does (because of condensation!).

As a result, a profile can also be **conditionally unstable**, i.e., stable to dry motions but unstable to saturated motions.

Below, I've written you a function called `lift_moist_adiabatically`. This works identically to the `lift_dry_adiabatically` function from earlier, but assuming the parcel is saturated and follows a moist adiabat. 

In [ ]:
def lift_moist_adiabatically(T, p, p_new):
    # Calculate the new temperature (T_new) [degC] as a function of initial pressure (p) [hPa], final pressure (p_new), and initial temperature (T) [degC] for moist adiabatic lifting

    import metpy.calc as mpcalc
    from metpy.units import units

    T_new = mpcalc.moist_lapse(
        p_new * units.hPa, T * units.degC, reference_pressure=p * units.hPa
    )
    return T_new.magnitude

Use the two functions to lift your parcel at 650 hPa from earlier both dry adiabatically and moist adiabatically.

In [ ]:
T_parcel_dry = lift_dry_adiabatically(T_init, p_init, 650)
T_parcel_saturated = lift_moist_adiabatically(T_init, p_init, 650)

Now we'll plot all three points (original parcel and two lifted parcels) on the Skew-T. I've zoomed in the Skew-T below so you can see it more clearly.

In [ ]:
# intialize figure
fig = plt.figure(figsize=(9, 9))

# initialize the Skew-T axis
skew = SkewT(fig, rotation=45)

# plot temperature and pressure on Skew-T
skew.ax.plot(df.temperature, df.pressure, color="black")

# original parcel
skew.ax.scatter(T_init, p_init, zorder=10, label="Original Parcel")

# lifted dry parcel
skew.ax.scatter(T_parcel_dry, p_parcel, zorder=10, label="Dry Lifted Parcel")

# lifted saturated parcel
skew.ax.scatter(
    T_parcel_saturated, p_parcel, zorder=10, label="Saturated Lifted Parcel"
)

# axis labels
skew.ax.set_xlabel("Temperature (°C)")
skew.ax.set_ylabel("Pressure (hPa)")

# adding temperature labels at the top of our plot to help read values
skew.ax.tick_params(
    axis="x",
    top=True,
    labeltop=True,
    bottom=True,
    labelbottom=True,
)

skew.ax.set_ylim(900, 500)
skew.ax.set_xlim(5, 30)

skew.ax.legend()

plt.show()
plt.close("all")

### Question 16 (2 pt)

Which cools more with height as it is lifted, the dry parcel or the saturated parcel? Explain physically why this is the case.

[1 pt for reading plot, 1 pt for explanation]

#### Answer 16

#TODO: *(Type your answer here)*

### Question 17 (3 pts)

Now, compare the dry (orange) and saturated (green) parcels to the environmental temperature at the same level. 

1. Is the sign of their temperature perturbation relative to the environment the same?
2. What does this mean for the directions of the buoyant force of the two lifted parcels?
3. Determine whether the profile at 700 hPa is stable/unstable/neutral to dry *and* saturated motions (one answer for dry and one answer for saturated). 

[1 pt each]

#### Answer 17

#TODO: *(Type your answer here)*

You have just described **conditional instability**! This means that the static stability of a parcel is *conditional* (i.e., depends on) whether or not it is saturated at that level.

Lastly, we can use a Skew-T with dry and moist adiabats to graphically determine conditional instability. In this case, we compare our environmental profile to both types of adiabat. 

In [ ]:
# intialize figure
fig = plt.figure(figsize=(9, 9))

# initialize the Skew-T axis
skew = SkewT(fig, rotation=45)

# plot temperature and pressure on Skew-T
skew.ax.plot(df.temperature, df.pressure, color="black")


# axis labels
skew.ax.set_xlabel("Temperature (°C)")
skew.ax.set_ylabel("Pressure (hPa)")

# adding temperature labels at the top of our plot to help read values
skew.ax.tick_params(
    axis="x",
    top=True,
    labeltop=True,
    bottom=True,
    labelbottom=True,
)


# draw dry adiabats for reference
skew.plot_dry_adiabats(np.arange(-30, 90, 5) * units("degC"))

# draw moist adiabats for reference
skew.plot_moist_adiabats(np.arange(-30, 50, 2.5) * units("degC"))

skew.ax.set_xlim(-15, 35)

plt.show()
plt.close("all")

### Question 18 (2 pts)

Use the Skew-T above to find the following:

1. one other pressure level where the profile is conditionally unstable (environmental lapse rate is between the dry and moist adiabatic lapse rates)
2. one pressure level where the profile is **absolutely** stable (environmental lapse rate is less than the moist adiabatic lapse rate)

#### Answer 18

#TODO: *(Type your answer here)*